In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load with latin-1 encoding — required for special characters like Ü, é, ñ in player names
df_2526 = pd.read_csv('../raw/players_data-2025_2026.csv', encoding='latin-1')
df_2324 = pd.read_csv('../raw/players_data-2023_2024.csv', encoding='latin-1')
df_hist = pd.read_csv('../raw/players_data-2014_2025.csv', encoding='latin-1')

print(f"2025-26 season: {len(df_2526)} players")
print(f"2023-24 season: {len(df_2324)} players")
print(f"Historical 2014-2025: {len(df_hist)} rows across {df_hist['season'].nunique()} seasons")
print(f"\nHistorical seasons: {sorted(df_hist['season'].unique())}")
print(f"Historical leagues: {list(df_hist['league_name'].unique())}")

2025-26 season: 2839 players
2023-24 season: 2852 players
Historical 2014-2025: 23607 rows across 11 seasons

Historical seasons: ['2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25']
Historical leagues: ['Premier League', 'Bundesliga', 'Serie A', 'Ligue 1']


In [3]:
def extract_nation_code(nation_str):
    """
    Converts 'br BRA' → 'BRA'
    Converts 'eng ENG' → 'ENG'
    """
    if pd.isna(nation_str):
        return None
    parts = str(nation_str).strip().split()
    if len(parts) >= 2:
        return parts[-1].upper()  # take the last part — the 3-letter code
    return nation_str.upper()

# Apply to both current season files
df_2526['nation_code'] = df_2526['Nation'].apply(extract_nation_code)
df_2324['nation_code'] = df_2324['Nation'].apply(extract_nation_code)

print("Top 20 nations in 2025-26 data:")
print(df_2526['nation_code'].value_counts().head(20))

Top 20 nations in 2025-26 data:
nation_code
ESP    419
FRA    332
GER    227
ITA    213
ENG    199
BRA     92
NED     82
ARG     81
POR     60
BEL     58
SEN     56
DEN     54
MAR     51
CIV     48
SUI     47
SWE     36
NGA     35
NOR     34
AUT     30
CRO     29
Name: count, dtype: int64


In [4]:
# FIFA World Cup 2026 qualified teams (48-team format)

WC_TEAMS = {

    # ================= UEFA (Europe) =================
    'ENG': 'England',
    'FRA': 'France',
    'ESP': 'Spain',
    'GER': 'Germany',
    'POR': 'Portugal',
    'NED': 'Netherlands',
    'BEL': 'Belgium',
    'CRO': 'Croatia',
    'SUI': 'Switzerland',
    'DEN': 'Denmark',
    'SWE': 'Sweden',
    'NOR': 'Norway',
    'SCO': 'Scotland',
    'TUR': 'Turkey',
    'CZE': 'Czechia',
    'BIH': 'Bosnia and Herzegovina',

    # ================= CONMEBOL (South America) =================
    'ARG': 'Argentina',
    'BRA': 'Brazil',
    'URU': 'Uruguay',
    'COL': 'Colombia',
    'ECU': 'Ecuador',
    'PAR': 'Paraguay',

    # ================= CONCACAF =================
    'USA': 'United States',
    'MEX': 'Mexico',
    'CAN': 'Canada',
    'PAN': 'Panama',
    'CUW': 'Curacao',
    'HAI': 'Haiti',

    # ================= CAF (Africa) =================
    'MAR': 'Morocco',
    'SEN': 'Senegal',
    'GHA': 'Ghana',
    'EGY': 'Egypt',
    'TUN': 'Tunisia',
    'CIV': "Cote d'Ivoire",
    'COD': 'DR Congo',
    'CPV': 'Cabo Verde',
    'DZA': 'Algeria',
    'RSA': 'South Africa',

    # ================= AFC (Asia) =================
    'JPN': 'Japan',
    'KOR': 'South Korea',
    'IRN': 'Iran',
    'SAU': 'Saudi Arabia',
    'AUS': 'Australia',
    'QAT': 'Qatar',
    'UZB': 'Uzbekistan',
    'JOR': 'Jordan',
    'IRQ': 'Iraq',

    # ================= OFC =================
    'NZL': 'New Zealand'
}

print(f"Total teams: {len(WC_TEAMS)}")

# Filter player data to only World Cup nations
wc_players_2526 = df_2526[df_2526['nation_code'].isin(WC_TEAMS.keys())].copy()
wc_players_2324 = df_2324[df_2324['nation_code'].isin(WC_TEAMS.keys())].copy()

print(f"\n2025-26 players from WC nations: {len(wc_players_2526)}")
print(f"2023-24 players from WC nations: {len(wc_players_2324)}")

print("\nNations with most players in current season data:")
nation_counts = wc_players_2526['nation_code'].value_counts()
for code, count in nation_counts.items():
    nation_name = WC_TEAMS.get(code, code)
    print(f"  {nation_name} ({code}): {count} players")

Total teams: 48

2025-26 players from WC nations: 2166
2023-24 players from WC nations: 2122

Nations with most players in current season data:
  Spain (ESP): 419 players
  France (FRA): 332 players
  Germany (GER): 227 players
  England (ENG): 199 players
  Brazil (BRA): 92 players
  Netherlands (NED): 82 players
  Argentina (ARG): 81 players
  Portugal (POR): 60 players
  Belgium (BEL): 58 players
  Senegal (SEN): 56 players
  Denmark (DEN): 54 players
  Morocco (MAR): 51 players
  Cote d'Ivoire (CIV): 48 players
  Switzerland (SUI): 47 players
  Sweden (SWE): 36 players
  Norway (NOR): 34 players
  Croatia (CRO): 29 players
  United States (USA): 28 players
  Ghana (GHA): 28 players
  Uruguay (URU): 26 players
  Japan (JPN): 22 players
  Colombia (COL): 19 players
  Turkey (TUR): 19 players
  Scotland (SCO): 18 players
  Czechia (CZE): 15 players
  DR Congo (COD): 14 players
  Bosnia and Herzegovina (BIH): 11 players
  Ecuador (ECU): 10 players
  Canada (CAN): 9 players
  South Kore

In [5]:
print("========== DATASET OVERVIEW ==========")

print(f"2025-26 shape: {df_2526.shape}")
print(f"2023-24 shape: {df_2324.shape}")
print(f"2014-25 shape: {df_hist.shape}")

print("\n========== COLUMN COUNTS ==========")

print(f"2025-26 columns: {len(df_2526.columns)}")
print(f"2023-24 columns: {len(df_2324.columns)}")
print(f"2014-25 columns: {len(df_hist.columns)}")

print("\n========== IMPORTANT COLUMN CHECK ==========")

important_cols = [
    'Player', 'Nation', 'Pos', 'Squad',
    'Age', 'Min', '90s',
    'Gls', 'Ast',
    'xG', 'xAG', 'xA',
    'Sh', 'SoT',
    'PrgP', 'PrgC',
    'Int', 'TklW',
    'GA', 'Save%'
]

for col in important_cols:
    print(
        f"{col:12} | "
        f"2025-26: {col in df_2526.columns} | "
        f"2023-24: {col in df_2324.columns} | "
        f"2014-25: {col in df_hist.columns}"
    )

print("\n========== CREATING MISSING COLUMNS ==========")

required_numeric_cols = [
    'xG', 'xAG', 'xA',
    'PrgP', 'PrgC',
    'Sh', 'SoT',
    'Int', 'TklW',
    'GA', 'Save%'
]

datasets = [df_2526, df_2324, df_hist]

for df in datasets:
    for col in required_numeric_cols:
        if col not in df.columns:
            df[col] = 0

print("Missing columns filled successfully.")

print("\n========== SAMPLE PLAYERS ==========")

sample_cols = [
    'Player',
    'nation_code',
    'Pos',
    'Squad',
    'Gls',
    'Ast',
    'xG',
    'PrgP',
    'Sh',
    'SoT',
    'Int',
    'TklW'
]

sample_cols = [c for c in sample_cols if c in wc_players_2526.columns]

sample = wc_players_2526.sort_values('Gls', ascending=False)

print(sample[sample_cols].head(15).to_string())

========== DATASET OVERVIEW ==========
2025-26 shape: (2839, 103)
2023-24 shape: (2852, 38)
2014-25 shape: (23607, 23)

========== COLUMN COUNTS ==========
2025-26 columns: 103
2023-24 columns: 38
2014-25 columns: 23

========== IMPORTANT COLUMN CHECK ==========
Player       | 2025-26: True | 2023-24: True | 2014-25: False
Nation       | 2025-26: True | 2023-24: True | 2014-25: False
Pos          | 2025-26: True | 2023-24: True | 2014-25: False
Squad        | 2025-26: True | 2023-24: True | 2014-25: False
Age          | 2025-26: True | 2023-24: True | 2014-25: False
Min          | 2025-26: True | 2023-24: True | 2014-25: False
90s          | 2025-26: True | 2023-24: True | 2014-25: False
Gls          | 2025-26: True | 2023-24: True | 2014-25: False
Ast          | 2025-26: True | 2023-24: True | 2014-25: False
xG           | 2025-26: False | 2023-24: True | 2014-25: True
xAG          | 2025-26: False | 2023-24: True | 2014-25: False
xA           | 2025-26: False | 2023-24: False | 2014-

In [6]:
print("========== MERGING CURRENT + ADVANCED STATS ==========")

# Keep only useful advanced columns from 2023-24
advanced_cols = [
    'Player',
    'nation_code',
    'xG',
    'xAG',
    'xA',
    'PrgP',
    'PrgC'
]

advanced_df = df_2324[advanced_cols].copy()

# Rename advanced stats
advanced_df = advanced_df.rename(columns={
    'xG': 'prev_xG',
    'xAG': 'prev_xAG',
    'xA': 'prev_xA',
    'PrgP': 'prev_PrgP',
    'PrgC': 'prev_PrgC'
})

# Merge with current season
merged_df = pd.merge(
    wc_players_2526,
    advanced_df,
    on=['Player', 'nation_code'],
    how='left'
)

# Fill NaNs after merge
merge_fill_cols = [
    'prev_xG',
    'prev_xAG',
    'prev_xA',
    'prev_PrgP',
    'prev_PrgC'
]

for col in merge_fill_cols:
    merged_df[col] = merged_df[col].fillna(0)

print(f"Merged dataset shape: {merged_df.shape}")

print("\nSample merged players:")
print(
    merged_df[
        [
            'Player',
            'nation_code',
            'Gls',
            'Ast',
            'prev_xG',
            'prev_PrgP'
        ]
    ]
    .head(15)
    .to_string()
)

========== MERGING CURRENT + ADVANCED STATS ==========
Merged dataset shape: (2248, 108)

Sample merged players:
                 Player nation_code  Gls  Ast  prev_xG  prev_PrgP
0      Brenden Aaronson         USA    4    5      2.0       56.0
1          Jerome Abbey         ENG    0    0      0.0        0.0
2           Zach Abbott         ENG    0    0      0.0        0.0
3   Jones El-Abdellaoui         MAR    2    0      0.0        0.0
4              Ali Abdi         TUN    3    0      0.0        0.0
5     Salis Abdul Samed         GHA    0    0      0.8       78.0
6       Laurent Abergel         FRA    1    0      1.1      194.0
7        Matthis Abline         FRA    6    4      3.8       20.0
8                 Abner         BRA    3    2      0.1       33.0
9     Zakaria Aboukhlal         MAR    0    0      2.8       20.0
10          Abdel Abqar         MAR    0    2      0.5       49.0
11        Tammy Abraham         ENG    2    0      0.6        4.0
12          Francis Abu      

In [7]:
def build_team_features(players_df, nation_code, min_minutes=500):

    team = players_df[
        (players_df['nation_code'] == nation_code)
    ].copy()

    # Ensure numeric
    numeric_cols = [
        'Min', '90s',
        'Gls', 'Ast',
        'Sh', 'SoT',
        'Int', 'TklW',
        'GA', 'Save%',
        'prev_xG',
        'prev_xAG',
        'prev_PrgP',
        'prev_PrgC'
    ]

    for col in numeric_cols:
        if col in team.columns:
            team[col] = pd.to_numeric(team[col], errors='coerce').fillna(0)

    # Filter minutes
    team = team[team['Min'] >= min_minutes]

    if len(team) == 0:
        return None

    # Safe 90s
    team['90s'] = team['90s'].clip(lower=0.1)

    # Per90 metrics
    team['goals_per90'] = team['Gls'] / team['90s']
    team['assists_per90'] = team['Ast'] / team['90s']
    team['shots_per90'] = team['Sh'] / team['90s']

    attackers = team[team['Pos'].str.contains('FW', na=False)]
    midfielders = team[team['Pos'].str.contains('MF', na=False)]
    defenders = team[team['Pos'].str.contains('DF', na=False)]

    features = {

        'nation': nation_code,
        'nation_name': WC_TEAMS.get(nation_code),

        # Current form
        'team_total_goals': team['Gls'].sum(),
        'team_total_assists': team['Ast'].sum(),

        # Attacking strength
        'attack_goals_per90':
            attackers['goals_per90'].mean()
            if len(attackers) > 0 else 0,

        # Historical xG form
        'historical_xg':
            team['prev_xG'].mean(),

        # Progressive play
        'historical_progressive_passing':
            team['prev_PrgP'].mean(),

        # Shooting
        'shots_per90':
            team['shots_per90'].mean(),

        # Defensive
        'avg_interceptions':
            team['Int'].mean(),

        'avg_tackles_won':
            team['TklW'].mean(),

        # Goalkeeper quality
        'avg_save_pct':
            team['Save%'].mean(),

        # Squad depth
        'num_players_500min':
            len(team),

        'total_minutes':
            team['Min'].sum()
    }

    return features

In [8]:
print("========== BUILDING TEAM FEATURES ==========")

all_team_features = []

for nation in WC_TEAMS.keys():

    features = build_team_features(
        merged_df,
        nation
    )

    if features:
        all_team_features.append(features)

team_features_df = pd.DataFrame(all_team_features)

print(f"Built features for {len(team_features_df)} teams")

print(
    team_features_df[
        [
            'nation_name',
            'team_total_goals',
            'historical_xg',
            'avg_save_pct'
        ]
    ]
    .sort_values('team_total_goals', ascending=False)
    .to_string()
)

========== BUILDING TEAM FEATURES ==========
Built features for 41 teams
               nation_name  team_total_goals  historical_xg  avg_save_pct
2                    Spain               576       1.304530      3.919164
1                   France               562       1.491743      3.163303
3                  Germany               420       1.677844      6.541317
0                  England               374       2.519847      2.551145
17                  Brazil               229       1.970667      4.016000
16               Argentina               177       2.103125      6.725000
4                 Portugal               144       1.679245      3.824528
5              Netherlands               142       1.817910      5.332836
7                  Croatia                89       2.173913      2.921739
9                  Denmark                86       0.997222      5.355556
28                 Senegal                86       1.317500      4.965000
32           Cote d'Ivoire             

In [11]:
print("========== WORLD CUP POWER RANKINGS ==========")

ranked = team_features_df.sort_values(
    'team_total_goals',
    ascending=False
)

for i, row in enumerate(ranked.itertuples(), start=1):

    bar = '█' * int(row.team_total_goals / 5)

    print(
        f"{i:2}. "
        f"{row.nation_name:20} "
        f"{int(row.team_total_goals):4} goals "
        f"{bar}"
    )

========== WORLD CUP POWER RANKINGS ==========
 1. Spain                 576 goals ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████
 2. France                562 goals ████████████████████████████████████████████████████████████████████████████████████████████████████████████████
 3. Germany               420 goals ████████████████████████████████████████████████████████████████████████████████████
 4. England               374 goals ██████████████████████████████████████████████████████████████████████████
 5. Brazil                229 goals █████████████████████████████████████████████
 6. Argentina             177 goals ███████████████████████████████████
 7. Portugal              144 goals ████████████████████████████
 8. Netherlands           142 goals ████████████████████████████
 9. Croatia                89 goals █████████████████
10. Denmark                86 goals █████████████████
11. Senegal                8